# RunPod Training Notebook: Hana Kim (EA-F-001) Flux.1-dev LoRA
This notebook automates setting up the workspace on RunPod and running the LoRA training using `ostris/ai-toolkit` for Flux.1-dev.

### Step 1: Install Dependencies
Run this cell to clone the repository and install the requirements.

In [ ]:
# Clone the trainer repo
!git clone --recursive https://github.com/ostris/ai-toolkit.git /workspace/ai-toolkit
%cd /workspace/ai-toolkit

# Install python requirements
!pip install -r requirements.txt
!pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu121

### Step 2: Upload Dataset
Please upload the `character_id` directory to `/workspace/character_id`. Alternatively, run this cell to create the directories and paste your dataset here.

In [ ]:
import os
os.makedirs("/workspace/character_id/train", exist_ok=True)
os.makedirs("/workspace/character_id/identity_anchors", exist_ok=True)
print("Created directories on RunPod. Upload your images and captions to /workspace/character_id/train/")

### Step 3: Create Training Configuration
Run this cell to write the `flux_training_config.yaml` file needed for `ai-toolkit`.

In [ ]:
yaml_config = """
job: extension
config:
  name: "ea_f_001_hana_kim_lora"
  process:
    - type: 'sd_trainer'
      training_folder: "/workspace/output"
      performance_log_every: 250
      device: cuda
      trigger_word: "ea_f_001_hana_kim"
      network:
        type: "lora"
        linear: 16
        linear_alpha: 16
      save:
        dtype: float16
        save_every: 500
        max_step_saves_to_keep: 4
      datasets:
        - folder_path: "/workspace/character_id/train"
          caption_ext: "txt"
          resolution: [512, 768, 1024]
      train:
        epochs: 45
        steps: 2500
        gradient_accumulation_steps: 1
        learning_rate: 1e-4
        batch_size: 1
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        optimizer: "adamw8bit"
        ema_decay: 0.99
        dtype: bf16
      model:
        name_or_path: "black-forest-labs/FLUX.1-dev"
        is_flux: true
        quantize: true
      sample:
        sampler: "flowmatch"
        sample_every: 250
        width: 1024
        height: 1024
        prompts:
          - "ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, wearing elegant red halterneck dress, outdoor garden background, high fashion editorial, 85mm portrait"
          - "ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, wearing cream sleeveless knit dress, luxury studio background, direct gaze, soft lighting"
"""

with open("/workspace/ai-toolkit/config/flux_training_config.yaml", "w") as f:
    f.write(yaml_config)
print("Training configuration written to /workspace/ai-toolkit/config/flux_training_config.yaml")

### Step 4: Run Training
Run this cell to start the Flux.1-dev LoRA training job. This requires a GPU with at least 24GB VRAM (e.g., RTX 3090, RTX 4090, or A100).

In [ ]:
%cd /workspace/ai-toolkit
!python run.py config/flux_training_config.yaml

### Step 5: Export Model
Once training completes, your LoRA weights `.safetensors` will be stored in `/workspace/output/ea_f_001_hana_kim_lora`. You can download it directly from RunPod's files manager or copy it to storage.